In [35]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndaijwm/Documents/deidentification/deidentification/deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task
from qc_package.scanner import DbScanner
import pandas as pd

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [36]:
TableDetailsModel.objects.filter(table_status=2).count()

44

In [37]:
qc_config = {
    "PATIENT_ID": {"prefix_value": "1001000", "length_of_value": 15},
    "ENCOUNTER_ID": {"prefix_value": "1001000", "length_of_value": 19},
}
db = DbDetailsModel.objects.get(db_name="jwm_deidentification")

db_scanner = DbScanner(
    db.source_db_config['connection_str'],
    db.destination_db_config['connection_str'],
    db.run_config["mapping_db_config"],
    qc_config
)

In [ ]:
input_csv = "northwest_phi.csv"
df = pd.read_csv(input_csv)
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

In [ ]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

In [ ]:
tables_for_qc = ['additionalnotes']
len(tables_for_qc)

In [38]:
tables_for_qc = ['br_ServiceDetailHistory',
'OBXManualHistory',
'OBXManual',
'OBX',
'PatMedRecon',
'br_VisitDiagnosis',
'br_ServiceDetail',
'OrdReqOrderOrdReqDiagnosis',
'OrdReqClinicalDocuments',
'OrdReq',
'MedicationRxNormFact',
'InsuranceProviders',
'TypeOfService',
'NoteType',
'InsurancePlanClass',
'Insurance',
'OrdReqDiagnosisStatus']

tables_for_qc = ['br_ServiceDetailHistory',
'OBXManualHistory',
'OBXManual',
'OBX',
'PatMedRecon',
'br_VisitDiagnosis', 'OrdReqClinicalDocuments',
'MedicationRxNormFact']
print(len(tables_for_qc))

8


In [ ]:
code_failure = []
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    if table_obj.table_name in tables_for_qc:
        try:
            table_config = table_obj.table_details_for_ui
            output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
            table_obj.qc_result = output_result
            table_obj.save()
        except Exception as e:
            raise e
            code_failure.append(table_obj.table_name)

In [15]:
len(code_failure)

4

In [20]:
tb_obj = TableDetailsModel.objects.filter(table_name='OBXManualHistory')
tb_obj.output_result

AttributeError: 'QuerySet' object has no attribute 'output_result'

In [16]:
import os, json
# table_name, source_row_count, dest_row_count, qc_status, ignore_rows_count, reason, is_phi
all_results = []

def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    if table_obj.table_name in tables_for_qc:
        qc_result = table_obj.qc_result
        if not qc_result:
            continue
        one_table = {
            "table_name": table_obj.table_name,
            "source_rows_count": qc_result['source_rows_count'],
            "dest_rows_count": qc_result['dest_rows_count'],
            "qc_status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
            "ignore_rows_count": qc_result['ignore_rows_count'],
            "reason": qc_result.get("final_qc_result", {}).get("reason"),
            "is_phi": is_phi_table(table_obj.table_details_for_ui)
        }
        all_results.append(one_table)
len(all_results)

0

In [17]:
res_df = pd.DataFrame(all_results)
res_df.shape

(0, 0)

In [18]:
res_df.to_csv("qc_results_07022025.csv", index=False)